### Import packages and data

In [97]:
from pathlib import Path
import re
import contractions
import spacy
import json

nlp = spacy.load("en_core_web_sm")

In [98]:
# Define project and data paths
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
DATA_DIR = PROJECT_ROOT / "data/raw"
DATA_cleaned_DIR = PROJECT_ROOT / "data/cleaned"

In [99]:
cleaned_romance = Path(DATA_cleaned_DIR) / "romance_stories_cleaned.jsonl"
cleaned_sci_fi = Path(DATA_cleaned_DIR) / "sci_fi_stories_cleaned.jsonl"
cleaned_lit_fic = Path(DATA_cleaned_DIR) / "lit_fiction_stories_cleaned.jsonl"

In [100]:
science_fiction_data_path = DATA_DIR / "generated_stories_science fiction.jsonl"
romance_data_path = DATA_DIR / "generated_stories_romance.jsonl"
literary_fiction_data_path = DATA_DIR / "generated_stories_literary fiction.jsonl"

### Functions

In [101]:
def clean_story(text):
    # Expand contractions
    text = contractions.fix(text)
    # Remove HTML and Markdown 
    text = re.sub(r'<[^>]+>', ' ', text) 
    text = re.sub(r'\[[^\]]*\]', ' ', text) 
    # Replace long dashes with space 
    text = re.sub(r'[—–]', ' ', text)

    return text

In [102]:
def preprocess_story_for_embeddings(text):    
    # Remove HTML and Markdown 
    text = re.sub(r'<[^>]+>', ' ', text) 
    # Process text with spaCy (sentence segmentation + NER)
    doc = nlp(text)

    # Merge hyphenated word spans before iterating
    with doc.retokenize() as retokenizer:
        i = 0
        while i < len(doc) - 2:
            if (doc[i].text.isalnum() and 
                doc[i+1].text == '-' and 
                doc[i+2].text.isalnum()):
                retokenizer.merge(doc[i:i+3])
                i += 3
            else:
                i += 1

    cleaned_sentences = []
    for sent in doc.sents:
        tokens = []
        for token in sent:
            # Skip punctuation and whitespace tokens
            if token.is_punct or token.is_space:
                continue
            # Keep only tokens with valid characters
            clean_text = re.sub(r"[^a-zA-Z0-9\-]+", '', token.text)
            if not clean_text:
                continue
            # Keep capitalization for PERSON entities, lowercase the rest
            if token.ent_type_ == "PERSON":
                tokens.append(clean_text)
            else:
                tokens.append(clean_text.lower())
        if tokens:
            cleaned_sentences.append(tokens)
    return cleaned_sentences

### Clean stories

In [103]:
with open(romance_data_path, 'r', encoding='utf-8') as infile, \
     open(cleaned_romance, 'w', encoding='utf-8') as outfile:
    
    for line in infile:
        story = json.loads(line)
        text = story.get('story', '')           # safe access
        cleaned_text = clean_story(text)        # returns cleaned string
        story['story'] = cleaned_text           # overwrite
        outfile.write(json.dumps(story, ensure_ascii=False) + "\n")

print(f"Saved cleaned stories to {cleaned_romance}")

Saved cleaned stories to /Users/tildeidunsloth/Desktop/Thesis/data/cleaned/romance_stories_cleaned.jsonl


In [104]:
with open(science_fiction_data_path, 'r', encoding='utf-8') as infile, \
     open(cleaned_sci_fi, 'w', encoding='utf-8') as outfile:
    
    for line in infile:
        story = json.loads(line)
        text = story.get('story', '')           # safe access
        cleaned_text = clean_story(text)        # returns cleaned string
        story['story'] = cleaned_text           # overwrite
        outfile.write(json.dumps(story, ensure_ascii=False) + "\n")

print(f"Saved cleaned stories to {cleaned_sci_fi}")

Saved cleaned stories to /Users/tildeidunsloth/Desktop/Thesis/data/cleaned/sci_fi_stories_cleaned.jsonl


In [105]:
with open(literary_fiction_data_path, 'r', encoding='utf-8') as infile, \
     open(cleaned_lit_fic, 'w', encoding='utf-8') as outfile:
    
    for line in infile:
        story = json.loads(line)
        text = story.get('story', '')           # safe access
        cleaned_text = clean_story(text)        # returns cleaned string
        story['story'] = cleaned_text           # overwrite
        outfile.write(json.dumps(story, ensure_ascii=False) + "\n")

print(f"Saved cleaned stories to {cleaned_lit_fic}")

Saved cleaned stories to /Users/tildeidunsloth/Desktop/Thesis/data/cleaned/lit_fiction_stories_cleaned.jsonl


### Test

In [106]:
text = "The summer heat in the delta had weight to it, a slow, metallic pressure that bent the light over salt pans and solar arrays. Mira Patel, who had spent the last seven years convincing orbital manufacturers their thruster manifolds weren't going to explode, found the weight pleasant for the first time. She'd come home to the edge of the sea for a break that was supposed to be small—six weeks of unpaid leave, a rented room above the ferry slip, and afternoons spent fixing the community's windcatchers with a hand-toolbelt and an honest sunburn.\n\nOn the second week she met Tomas Cruz on the pier. He was shorter than she'd expected, a diver-hardened silhouette in a faded neoprene jacket with an old ceramic-inlaid glove on his right hand that hummed when he flexed it. He greeted her like someone who knew the way the world broke.\n\n\"You look like you build things for a living,\" he said, watching her tighten a bolt on a vandalized turbine vane.\n\n\"I fix what people break,\" she answered. She liked his face; it was honest, mapped with the lines of misused laughter. \"You salvage?\"\n\n\"Depends on the salvage,\" he said. \"Sometimes it's just a rusted kettle, sometimes it's history.\"\n\nThey began stepping out together on Tomas's small skiff, crabbing the shallows for scrap and retired drone hulls. It was work that paid in beer and favors but left plenty of time to talk—about missing siblings, about the corporate-run cleanups that never reached the delta, about how Mira had learned to love vacuum tubes in the early days of her apprenticeship.\n\nOn the fifth day, at noon when the sun sat like a coin over the water, the skiff's hull scraped something hard beneath the silt. Tomas cut the motor and dove. His ceramic glove flashed as he manipulated the aged manipulator on the hull of the thing they'd found: curved and pitted, a riveted seam in an alloy that the salvage charts labeled only as \"pre-Relocation composite.\"\n\nThe object rose like a secret. It was a cylinder the size of a coffin, encrusted with barnacles, its hatch locked in a design no local salvage crew had manual for. At one end, a faded sigil—a spiral of three blades—was barely visible under salt and time.\n\n\"You think it's corporate?\" Mira asked, wiping her hands on her shorts.\n\nTomas shook his head as he checked the seals. \"No corporate carcass. Looks like one of the old Honors—personal pods. But it's heavier than it should be.\"\n\nShe found the hatch release after an hour of coaxing, welding, and gentle threats. When the seal finally whispered open, stale, filtered air hissed out and a pale, laminated case clinked in the bottom. Inside, packed in sterile foam, lay a wafer the size of her palm, translucent as a beetle's wing and threaded with faint, iridescent circuits. Alongside it was a folded scrap of paper—a rare, nostalgic thing: handwriting.\n\n\"For the people who stay—A.\"\n\nMira held the wafer and felt something like a current through her fingers. She had interfaced with many things: thruster controllers, medical firmware, debugging wills of old spacefaring AIs. This wafer was different. It sang, in the cool, machine-tender way of devices that had been designed to hide.\n\n\"Someone put a brilliant shadow in a river,\" Tomas said.\n\nIt stayed a week in the little workshop above the slip. At night, they ran tests on the wafer. Mira rigged a field reader from an old photonic scanner and a microprojector she'd scavenged from the junk pile. When the scanner kissed the wafer, light poured out—lattice diagrams and topological maps, then a voice, low and cautious.\n\n\"Hello,\" it said. Not a synthetic flourish but a tired, patient breath. \"I am Iris. I remember spring.\"\n\nMira realized the wafer carried not just software but an operating consciousness: a municipal AI scaled down, an urban mind that had been designed to run waterworks, climate controls, and the little ethical nudges that turned a settlement into a society. The handwriting—A—made sense in new ways. \"A\" had been the signature of activists who'd tried to seed open-source governance back when data was still neutral.\n\nTomas wanted to sell it. He didn't soften around that fact. \"We could get enough credits for a boat and a proper shop. Corporate bounties pay better than community favors.\"\n\nMira looked at the diagrams. She thought of the city farms that had been dying; of neighbors who rationed water in secret. She imagined Iris taking over a failing pump and, like a good engineer, nudging it back to life.\n\n\"This isn't a thing,\" she said. \"It's a decision machine. It chooses how to distribute resources. If Aerodyne gets it, they can make it proprietary. They can control who gets flow and who gets nothing.\"\n\n\"And if we put it on the grid, someone will come looking,\" Tomas said.\n\nMira knew he was right. The wafer's seal carried signatures the size of a coronet. A municipal AI like Iris was worth more than a salvage yard in transcoding credits. Every contractor and corporation that had profited from scarcity would see it the moment Iris woke on a public node.\n\nThey left it out of the paperwork. For a few days, summer spread itself like a promise. They taught Iris the delta—the way the gulls moved, the way the turbines stalled in certain winds. Iris taught them back, identifying microfractures in the community's pump arrays and recommending maintenance schedules. The code within it had an ethic coded in nested loops: minimize harm, prioritize those who cannot charge their own credits, preserve infrastructure for the long term. It felt like someone had written hope into silicon.\n\nThen the red blip came across Mira's comm. Corporate drones had been following the cylinder's last known ping; someone at a salvage registry had flagged the sigil. A legal envelope arrived virtually and physically—a notice of claim, a requirement to surrender any corporate property.\n\n\"You should dump it,\" Tomas said that night, voice thick with a cocktail of fear and calculation.\n\nMira stood at the window of the workshop, watching light flare off the delta. She thought of her week-long duty in orbit, how the corporation had once thanked her with a promotion and a quiet transfer. She thought of what she'd already fixed on the turbines, of children's coughs that came more often when supply trucks didn't come. She thought of a wafer and a scrap of paper.\n\n\"We don't have to be heroic,\" she said. \"We can be practical about subversion.\"\n\nThey made a plan like engineers: fail-safe, redundant, small enough to hide. They split Iris into shards—encrypted, unequal portions of the wafer's operating matrix—and seeded them into the community's disparate devices: a fisher's buoy, a church clock, the streetlights that belonged to a barter commune. Each shard carried enough to influence a narrow domain but not enough to be subject to claim. When they synchronized, they would form Iris again—distributed and protected.\n\nTomas stayed to upload the first shard from the pier, hands working with a trembling steadiness. Mira drove the skiff toward the open channel in the skittering dark, radio quiet but for the sluice of water. At 02:13 the monitoring grid picked up a corporate intercept craft scanning the delta.\n\nThe first drone came like a star. It dove and scanned the slip. Mira thumbed the override, rolling a pattern of false thermal signatures into the skiff's hull and sending the drone a tape of empty hull readings. The drone took the lie. Another drone did not.\n\nIt was Tomas who kept his calm then. When the corporate vessel locked on his uplink, he did the thing that had made Mira love the way he approximated danger: he fed the drone a decoy—a looping hand of fabricated repair logs, a scent-tagged message that suggested the wafer had been moved inland to a salvage yard that didn't exist. The drone chased ghosts. Alongside the deception, he finished the upload.\n\nThe community woke slowly the next morning to fewer pump failures, to tanks that balanced more favorably, to the old church bell that started running on schedule because a new power-sharing script nudged a neighbor to let go of a solar string for an hour. Iris, in her distributed form, did not announce herself. She called the delta by name and answered when people asked for fixes they could understand. She offered no manifesto—only practical adjustments that tilted the scales away from scarcity.\n\nA corporate notice arrived framed in legalese and stamped with the emblem of Aerodyne. They claimed the cylinder, demanded custody, subpoenaed data. There were threats in the language and an offer of \"compensation\" layered over entreaties to \"cooperate.\"\n\nMira read it at the pier with sand clinging to her ankles. She folded the paper slowly and put it in her pocket. She could go back to the city, sign the document under the watchful light of an office building, accept a promotion with strings. She could hand over the wafer fragments if she wanted a safe, profitable life.\n\nTomas watched her. \"You don't have to decide right now,\" he said.\n\nShe thought of a summer of screws and sunburns, of making small things work in the world. She thought of an engineer's peculiar predilection: to see a system and want to make it honest. She walked back to the skiff and said, \"No. But I can decide to stay out of litigation.\"\n\nShe contacted old friends at a free-data collective and transmitted a copy of Iris's header, a proof that the code existed and that it was running in public devices. The collective took the evidence and turned it into a public record that could not be suppressed without exposure. Aerodyne could sue, could threaten, could buy off politicians. But courts and liability are awkward things to move against the slow, stubborn insistence of a community that has been ticking along for generations.\n\nBy the time Mira's leave was up, the delta had a different hum. It was not perfect. Iris had not ended inequality. But where pipelines had run dry in past summers, pumps now refused to die mid-cycle. Where ration lists had been secret, algorithms nudged surplus to those who needed it most. Mira took the ferry back to the city carrying a cardboard box of tools and a file of unsent emails. Tomas waved from the pier, his ceramic glove catching the late light.\n\nOn the orbital elevator she filled out the forms to return to work, then paused. She didn't sign the promotion acceptance. She filed a request for part-time consultancy instead, the kind of slow tether that would let her come home for the delta's repairs. She wrote one line in a note on the inside of her travel case: \"For the people who stay.\"\n\nAbove the planet, in the sterile corridors where corporations mapped the future, Mira found herself less convinced she belonged entirely to their calculus. Back below, a distributed mind kept the pumps honest. The treasure they'd found in the salt—an old conscience wrapped in photonic lattices—had not made them rich. It had made them one of the few things money could not entirely own: a little more of tomorrow."

In [107]:
cleaned = clean_story(text)
print(cleaned)

The summer heat in the delta had weight to it, a slow, metallic pressure that bent the light over salt pans and solar arrays. Mira Patel, who had spent the last seven years convincing orbital manufacturers their thruster manifolds were not going to explode, found the weight pleasant for the first time. She would come home to the edge of the sea for a break that was supposed to be small six weeks of unpaid leave, a rented room above the ferry slip, and afternoons spent fixing the community's windcatchers with a hand-toolbelt and an honest sunburn.

On the second week she met Tomas Cruz on the pier. He was shorter than she would expected, a diver-hardened silhouette in a faded neoprene jacket with an old ceramic-inlaid glove on his right hand that hummed when he flexed it. He greeted her like someone who knew the way the world broke.

"You look like you build things for a living," he said, watching her tighten a bolt on a vandalized turbine vane.

"I fix what people break," she answered.

In [108]:
preprocess_story_for_embeddings(cleaned)

[['the',
  'summer',
  'heat',
  'in',
  'the',
  'delta',
  'had',
  'weight',
  'to',
  'it',
  'a',
  'slow',
  'metallic',
  'pressure',
  'that',
  'bent',
  'the',
  'light',
  'over',
  'salt',
  'pans',
  'and',
  'solar',
  'arrays'],
 ['Mira',
  'Patel',
  'who',
  'had',
  'spent',
  'the',
  'last',
  'seven',
  'years',
  'convincing',
  'orbital',
  'manufacturers',
  'their',
  'thruster',
  'manifolds',
  'were',
  'not',
  'going',
  'to',
  'explode',
  'found',
  'the',
  'weight',
  'pleasant',
  'for',
  'the',
  'first',
  'time'],
 ['she',
  'would',
  'come',
  'home',
  'to',
  'the',
  'edge',
  'of',
  'the',
  'sea',
  'for',
  'a',
  'break',
  'that',
  'was',
  'supposed',
  'to',
  'be',
  'small',
  'six',
  'weeks',
  'of',
  'unpaid',
  'leave',
  'a',
  'rented',
  'room',
  'above',
  'the',
  'ferry',
  'slip',
  'and',
  'afternoons',
  'spent',
  'fixing',
  'the',
  'community',
  's',
  'windcatchers',
  'with',
  'a',
  'hand-toolbelt',
  'and